In [ ]:
import numpy as np 
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt                               
import os 
import gc
import matplotlib.colors as mcolors
from matplotlib.ticker import MultipleLocator
import matplotlib.ticker as ticker
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.gridspec as gridspec
matplotlib.rcParams["figure.dpi"] = 150
from particle import PDGID

In [ ]:
datadir = '/home/mwells5/Muon_Collider_Smart_Pixels/Data_Files/Data_Set_2026Feb_copy_m/Parquet_Files/'
trackDir = '/home/mwells5/Muon_Collider_Smart_Pixels/Data_Files/Data_Set_2026Feb_copy_m/tl_with_moduleID_07_28_2026/'
flp = 0

In [ ]:
truthbib = pd.DataFrame()
recon2Dbib = pd.DataFrame()
truthsig = pd.DataFrame()
recon2Dsig = pd.DataFrame()
#trackData = pd.DataFrame()
recon3Dsig = pd.DataFrame()
recon3Dbib = pd.DataFrame()

#test = pd.DataFrame()
#test = pd.read_parquet("/home/mwells5/Muon_Collider_Smart_Pixels/Data_Files/Data_Set_2026Feb_copy_m/Parquet_Files/bib_mp_labels_42.parquet")

labels_bib_list = []
recon2D_bib_list = []
labels_sig_list = []
recon2D_sig_list = []
recon3D_bib_list = []
recon3D_sig_list = []
#trackdata_list = []

#trackHeader = ["cota", "cotb", "p", "flp", "ylocal", "zglobal", "pt", "t", "hit_pdg", "moduleID"]

count=0
for file in os.listdir(datadir):
    if "labels" in file:
        if "bib" in file: 
            labels_bib_list.append(pd.read_parquet(f"{datadir}{file}")) 
            #truthbib = pd.concat([truthbib,pd.read_parquet(f"{datadir}{file}")])
            file = file.replace("labels","recon2D")
            recon2D_bib_list.append(pd.read_parquet(f"{datadir}{file}")) 
            #recon2Dbib = pd.concat([recon2Dbib,pd.read_parquet(f"{datadir}{file}")])
            file = file.replace("recon2D","recon3D")
            recon3D_bib_list.append(pd.read_parquet(f"{datadir}{file}"))
        elif "sig" in file: 
            labels_sig_list.append(pd.read_parquet(f"{datadir}{file}")) 
            #truthsig = pd.concat([truthsig,pd.read_parquet(f"{datadir}{file}")])
            file = file.replace("labels","recon2D")
            recon2D_sig_list.append(pd.read_parquet(f"{datadir}{file}")) 
            #recon2Dsig = pd.concat([recon2Dsig,pd.read_parquet(f"{datadir}{file}")])
            file = file.replace("recon2D","recon3D")
            recon3D_sig_list.append(pd.read_parquet(f"{datadir}{file}"))
            count+=1
        if count == 100:
            break

# countmm=0
# countmp=0
# for file in os.listdir(trackDir):
#     if "bib_mm" in file:
#         trackdata_list.append(pd.read_csv(f"{trackDir}{file}", sep=' ', names=trackHeader))
#         countmm+=1 
#     elif "bib_mp" in file: 
#         trackdata_list.append(pd.read_csv(f"{trackDir}{file}", sep=' ', names=trackHeader))
#         countmp+=1
#     if countmp+countmm==200:
#         break 
            

truthbib = pd.concat(labels_bib_list)
recon2Dbib = pd.concat(recon2D_bib_list)
truthsig = pd.concat(labels_sig_list)
recon2Dsig = pd.concat(recon2D_sig_list)
recon3Dbib = pd.concat(recon3D_bib_list)
recon3Dsig = pd.concat(recon3D_sig_list)
#trackData = pd.concat(trackdata_list)

del labels_bib_list
del recon2D_bib_list
del labels_sig_list
del recon2D_sig_list
del recon3D_sig_list
del recon3D_bib_list
#del trackdata_list

gc.collect()

#clustersSig = recon2Dsig.to_numpy().reshape(recon2Dsig.shape[0],13,21)
#clustersBib = recon2Dbib.to_numpy().reshape(recon2Dbib.shape[0],13,21)

#trackData['adjusted_hit_time'] = trackData['t']-1e6*np.sqrt(trackData['zglobal']**2+30**2)/299792458

clustersSig_time = recon3Dsig.to_numpy().reshape(recon3Dsig.shape[0],20,13,21)
clustersBib_time = recon3Dbib.to_numpy().reshape(recon3Dbib.shape[0],20,13,21)
print(clustersBib_time[0])

print(f"# of bib clusters: {len(truthbib)}\n# of sig clusters {len(truthsig)}")
print(f"Total # of clusters: {len(truthbib)+len(truthsig)}")
print(f"keys bib {recon2Dbib.keys()} ")
print(f"and of truth {truthbib.keys()}")
print(truthbib.head())
# print(f"length of tracklist {len(trackData)}")
# print(f"tracklist keys {trackData.keys()}")
# print(trackData.head())
#print(truthbib['z-global'].iat[21])
#print(f"keys of recon3D {recon3Dbib.keys()}")

In [ ]:
def cut_data(data_df, recon_df):
    recon_df = recon_df[(data_df['z-global'] >= 0) & (data_df['z-global'] <= 13)]
    data_df = data_df[(data_df['z-global'] >= 0) & (data_df['z-global'] <= 13)]

    recon_df = recon_df[(data_df['adjusted_hit_time'] >= -.09) & (data_df['adjusted_hit_time'] <= .15)]
    data_df = data_df[(data_df['adjusted_hit_time'] >= -.09) & (data_df['adjusted_hit_time'] <= .15)]
    
    #recon_df = recon_df[data_df['moduleID'] == 1]
    #data_df = data_df[data_df['moduleID'] == 1]
    
    recon_df.reset_index()
    data_df.reset_index()

In [ ]:
def populate_module_time(data_df, recon_df):

    cut_data(data_df, recon_df)
    
    # create cluster frames
    clusters = recon_df.to_numpy().reshape(recon_df.shape[0],20,13,21)

    x_l = (data_df['z-global']%13)*40 # 1mm = 40px
    x_l = x_l.to_numpy()
    x_l = x_l.astype(int)

    y_l = ((data_df['y-local'])+8.5)*40 # changes range from 0 to 13 and convert to px
    y_l = y_l.to_numpy()
    y_l = y_l.astype(int)

    h_t = (data_df['hit_time'])
    h_t = h_t.to_numpy()

    t_start = data_df['hit_time'].min() 
    t_slice = 200 #picoseconds
    t_end = data_df['hit_time'].max() 

    frame_number = (t_end-t_start)/t_slice
    dec = str(frame_number).split('.')[1]

    if (int(dec[0])<5):
         frame_number = int(frame_number+1)
    else:
         frame_number = int(frame_number)

    mod_array_list = np.zeros((frame_number, 520, 520))

    timestamps = []
    j=0
    for j in range(frame_number):
         timestamps.append((j*t_slice)+t_start)

    i=0
    for i in range(recon_df.shape[0]):
      # defines time range for a hit
      t_min = format(h_t[i], 'f')
      dec2 = int(t_min.split('.')[0])

      if float(t_min)<0:
           start_frame_id = 0
      else:
           start_frame_id = int(dec2%t_slice)

      end_frame_id = start_frame_id+20

      if (start_frame_id>frame_number):
           continue
      elif (end_frame_id>frame_number):
           end_frame_id = frame_number
           clustersBib_time[i, :(end_frame_id+1)-(start_frame_id+1)]

      cluster_end = 20-(end_frame_id-start_frame_id)

      # defines x and y range on module
      cx_min = x_l[i] - 6
      cy_min = y_l[i] - 10
      cx_max = x_l[i] + 7
      cy_max = y_l[i] + 11
    
      # defines x and y range in cluster window
      wx_min = 0
      wy_min = 0 
      wx_max = 13
      wy_max = 21
    
      if cx_min < 0:
           wx_min -= cx_min
           cx_min = 0
      if cy_min < 0:
           wy_min -= cy_min
           cy_min = 0
      if cx_max > 520:
           wx_max -= (cx_max-520)
           cx_max = 520
      if cy_max > 520:
           wy_max -= (cy_max-520)
           cy_max = 520

      mod_array_list[start_frame_id:end_frame_id, cx_min:cx_max, cy_min:cy_max] += clusters[i, cluster_end:, wx_min:wx_max, wy_min:wy_max]

      if (frame_number>end_frame_id):
           extras = np.full_like(mod_array_list[end_frame_id:, cx_min:cx_max, cy_min:cy_max], clusters[i, 19, wx_min:wx_max, wy_min:wy_max])
           mod_array_list[end_frame_id:, cx_min:cx_max, cy_min:cy_max] += extras 
           
    return mod_array_list, timestamps

In [ ]:

def plotMod(mod_array, time):
    fig, ax = plt.subplots(figsize=(7,7),dpi=200)

    # Plot charge collected in each pixel
    #datamin = mod_array.min()
    #datamax = mod_array.max()
    im = ax.imshow(mod_array, 
                   #vmin=datamin, 
                   #vmax=datamax, 
                   cmap='magma_r', 
                   interpolation='nearest', norm='log')
    

    divider = make_axes_locatable(ax)
    cax = divider.append_axes('right', size='4%', pad=0.05)
    fig.colorbar(im, cax=cax, location='right',label='Number of eh pairs')
    ax.set_title("Charge for a z-slice of module width")

    divider = make_axes_locatable(ax)
    cax = divider.append_axes('right', size='4%', pad=0.05)
    fig.colorbar(im, cax=cax, location='right',label='Number of eh pairs')

    # Draw grid on both
    ax.set_xlim(0,520)
    ax.set_ylim(0,520)
    ax.set_xlabel("x-local [px]")
    ax.set_ylabel("y-local [px]")
    plt.figtext(0.525,0.2, f"Elapsed time: {time} ps", fontsize=12, 
                bbox=dict(facecolor='white', alpha=0.75))
    #plt.figtext(0.525,0.2, f"Occupied pixels: {percent_nonempty_pixels(module)}%", fontsize=12, 
                #bbox=dict(facecolor='white', alpha=0.75))
    ax.xaxis.set_major_locator(ticker.MultipleLocator(40))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(40))
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(20))
    ax.yaxis.set_minor_locator(ticker.MultipleLocator(20))
    plt.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)
    plt.tick_params(axis='y', which='both', left=False, right=False, labelleft=False)
    #ax.grid(which="minor", color="grey", linestyle='-', linewidth=0.2,snap=False)
    
    plt.tight_layout(pad=3.5)
    fig.canvas.draw()

#plotMod(module)
k=0
frames, times = populate_module_time(truthbib, recon3Dbib)
for k in range(3):
    plotMod(frames[k], times[k])
    #print(frames[k])

In [ ]:

totaltime_ns = 4 #nanoseconds
frame_number = 0
time_granulated_arrays = populate_module_time(truthbib, recon3Dbib)
i_x = 0
while i_x in range(frame_number+1):
    plotMod(time_granulated_arrays[i_x])
    i_x+=1